<a href="https://colab.research.google.com/github/gershonc/Ollama-Ngrok-Colab/blob/main/Running_ollama_in_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Free Ollama Colab Deploy

Welcome to the easiest way to deploy local Large Language Models (LLMs) for free!

This notebook demonstrates how to install and run **Ollama** directly within a Google Colab instance, and then securely expose the API using **ngrok**. This gives you a public API endpoint to interact with models like `qwen:0.5b`, `llama3`, and others from your local machine, applications, or anywhere in the world—completely free of charge.

### ✨ Features:
* **Zero-Cost Deployment:** Utilize Google Colab's free computing power.
* **Quick Setup:** Install and run Ollama in seconds.
* **Public API:** Seamlessly connect to your model via an ngrok public URL.

### 🛠️ Getting Started:
1. Run the setup cells to install Ollama.
2. Add your `NGROK_AUTH_TOKEN` in the Colab secrets tab (🔑).
3. Run the ngrok cell to get your public API link!

Let's get started! 👇

### 1. Install Ollama

First, we need to download and install Ollama. This will set up the necessary binaries on your Colab instance.

In [5]:
%%bash
sudo apt-get update
sudo apt-get install -y zstd
curl -fsSL https://ollama.com/install.sh | sh

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:5 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:6 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:7 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:9 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [88.7 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:11 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,533 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,898 kB]
Get:13 https://r2u.stat.il

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:114

### 2. Start the Ollama server

Ollama runs as a server process. We need to start it in the background so it can serve models.

In [7]:
import subprocess
import os
import time
import urllib.request
import urllib.error

# Set the OLLAMA_HOST environment variable to allow access from outside the server
os.environ['OLLAMA_HOST'] = '0.0.0.0'

# Start Ollama server in the background
# Use 'nohup' to keep it running even if the current shell exits
# Redirect output to a log file
ollama_process = subprocess.Popen(['nohup', 'ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, preexec_fn=os.setpgrp)

print("Ollama server started in the background.")
print("Waiting for the server to initialize...")

# Actively check if the server is ready
max_retries = 30
for i in range(max_retries):
    try:
        response = urllib.request.urlopen('http://localhost:11434/')
        if response.getcode() == 200:
            print("\nOllama server is ready!")
            break
    except urllib.error.URLError:
        print(".", end="", flush=True)
    time.sleep(1)
else:
    print("\nWarning: Ollama server did not respond in time. It might have failed to start.")

Ollama server started in the background.
Waiting for the server to initialize...

Ollama server is ready!


### 3. Pre-pull a recommended tiny model (e.g., `qwen:0.5b`)

This step downloads the model to your Colab instance, making subsequent uses faster as it won't need to download it again during `ollama run`.

In [8]:
%%bash
/usr/local/bin/ollama pull qwen:0.5b

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest 
pulling fad2a06e4cc7:   2% ▕                  ▏ 7.0 MB/394 MB                  pulling manifest 
pulling fad2a06e4cc7:  11% ▕█                 ▏  41 MB/394 MB                  pulling manifest 
pulling fad2a06e4cc7:  15% ▕██                ▏  59 MB/394 MB                  pulling manifest 
pulling fad2a06e4cc7:  23% ▕████              ▏  91 MB/394 MB                  pulling manifest 
pulling fad2a06e4cc7:  31% ▕█████             ▏ 121 MB/394 MB                  pulling manifest 
pulling fad2a06e4cc7:  35% ▕██████            ▏ 137 MB/394 MB                  pulling manifest 
pulling fad2a06e4cc7:  40% ▕███████           ▏ 159 MB/394 MB                  pulling manifest 
pulling fad2a06e4cc7:  49% ▕████████          ▏ 193 MB/394 MB                  pulling manifes

### 4. Run the `qwen:0.5b` model

Now you can run a prompt using the downloaded `qwen:0.5b` model. If you want to use `llama2` or another model, just replace `qwen:0.5b` with its name.

In [9]:
%%bash
/usr/local/bin/ollama run qwen:0.5b "What is the capital of France? Answer in one word."

Paris.



⠙ ⠙ ⠹ ⠸ ⠼ ⠦ ⠦ ⠧ ⠇ ⠏ ⠋ ⠹ ⠹ ⠸ ⠴ ⠦ ⠧ ⠇ ⠇ ⠋ ⠙ ⠹ ⠹ ⠼ ⠼ ⠦ ⠧ ⠧ ⠏ ⠏ ⠙ ⠙ ⠸ ⠸ ⠼ ⠦ ⠧ ⠧ ⠇ ⠋ ⠋ ⠹ ⠹ ⠼ ⠴ ⠴ ⠧ ⠇ ⠇ ⠋ ⠋ ⠙ ⠸ ⠸ ⠴ ⠴ ⠧ ⠇ ⠇ ⠋ ⠋ ⠹ ⠹ ⠼ ⠴ ⠦ ⠧ ⠇ ⠇ ⠋ ⠋ ⠙ ⠹ ⠼ ⠼ ⠦ ⠧ ⠧ ⠇ ⠋ ⠋ ⠙ ⠹ ⠼ ⠴ ⠴ ⠦ ⠧ ⠏ ⠏ ⠙ ⠙ ⠹ ⠼ ⠼ ⠴ ⠦ ⠧ ⠇ ⠋ ⠋ ⠙ ⠸ ⠸ ⠴ ⠦ ⠦ ⠧ ⠏ ⠋ ⠋ ⠙ ⠸ ⠼ ⠼ ⠴ ⠧ ⠧ ⠏ ⠏ ⠙ ⠹ ⠹ ⠼ ⠼ ⠦ ⠧ ⠧ ⠏ ⠏ ⠋ ⠙ ⠹ ⠼ ⠴ ⠴ ⠦ ⠇ ⠏ ⠋ ⠋ ⠙ ⠹ ⠼ ⠴ ⠴ ⠧ ⠇ ⠇ ⠋ ⠙ ⠙ ⠸ ⠼ ⠼ ⠴ ⠦ ⠧ ⠇ ⠋ ⠙ ⠙ ⠸ ⠸ ⠼ ⠴ ⠦ ⠇ ⠇ ⠏ ⠋ ⠙ ⠸ ⠸ ⠴ ⠦ ⠦ ⠇ ⠏ ⠏ ⠋ ⠙ ⠸ ⠸ ⠴ ⠴ ⠧ ⠇ ⠏ ⠏ ⠙ ⠙ ⠹ ⠼ ⠼ ⠴ ⠧ ⠇ ⠇ ⠋ ⠙ ⠹ ⠹ ⠼ ⠼ ⠦ ⠦ ⠇ ⠏ ⠏ ⠙ ⠙ ⠹ ⠼ ⠼ ⠴ ⠧ ⠇ ⠇ ⠋ ⠋ ⠙ ⠸ ⠼ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠙ ⠙ ⠹ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠙ ⠹ ⠼ ⠴ ⠴ ⠦ ⠧ ⠏ ⠋ ⠙ ⠹ ⠸ ⠸ ⠴ ⠴ ⠧ ⠧ ⠏ ⠏ ⠋ ⠹ ⠹ ⠼ ⠼ ⠦ ⠦ ⠇ ⠇ ⠏ ⠋ ⠙ ⠹ ⠸ ⠼ ⠴ ⠧ ⠧ ⠏ ⠏ ⠋ ⠙ ⠸ ⠸ ⠴ ⠴ ⠧ ⠧ ⠇ ⠏ ⠋ ⠙ ⠸ ⠸ ⠼ ⠦ ⠦ ⠇ ⠇ ⠋ ⠋ ⠹ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠹ ⠸ ⠸ ⠴ ⠴ ⠧ ⠇ ⠇ ⠏ ⠙ ⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠇ ⠇ ⠋ ⠋ ⠹ ⠸ ⠼ ⠼ ⠴ ⠦ ⠧ ⠏ ⠋ ⠋ ⠙ ⠸ ⠼ ⠼ ⠴ ⠧ ⠧ ⠏ ⠏ ⠋ ⠹ ⠹ ⠼ ⠴ ⠴ ⠧ ⠧ ⠏ ⠏ ⠙ ⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠇ ⠇ ⠏ ⠙ ⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠇ ⠇ ⠏ ⠋ ⠹ ⠸ ⠸ ⠴ ⠦ ⠦ ⠇ ⠇ ⠋ ⠋ ⠹ ⠹ ⠼ ⠼ ⠦ ⠦ ⠇ ⠇ ⠏ ⠋ ⠹ ⠹ ⠼ ⠴ ⠴ ⠧ ⠇ ⠇ ⠋ ⠙ ⠙ ⠹ ⠸ ⠴ ⠦ ⠧ ⠧ ⠇ ⠋ ⠋ ⠹ ⠸ ⠸ ⠴ ⠴ ⠦ ⠇ ⠏ ⠋ ⠙ ⠹ ⠸ ⠸ ⠴ ⠴ ⠧ ⠧ ⠏ ⠏ ⠙ ⠙ ⠹ ⠸ ⠼ ⠴ ⠧ ⠧ ⠏ ⠏ ⠙ ⠙ ⠸ ⠸ ⠼ ⠦ ⠦ ⠧ ⠏ ⠏ ⠙ ⠙ ⠸ ⠸ ⠴ ⠴ ⠧ ⠧ ⠏ ⠏ ⠙ ⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠹ ⠸ ⠴ ⠴ ⠧ ⠧ ⠇ ⠋ 

In [10]:
%%bash
ollama run qwen:0.5b "Why is the sky blue? Answer in one short sentence."

The sky is blue because of the scattering of sunlight by the Earth's atmosp
atmosphere. The amount of light that reaches the Earth's surface is about 3
360 million缕 (um) per second (s). This means that for every second that pa
passes, there are three to five times as many缕 of light that reach the Ear
Earth's surface. This scattering of sunlight by the Earth's atmosphere crea
creates a blue sky because the amount of light that reaches the Earth's sur
surface is about 360 million缕 (um) per second (s).



⠙ ⠙ 

### 5. Expose Ollama to your local PC using ngrok

To access the Ollama server from your local machine, we need to create a public tunnel to the Colab instance. We'll use `ngrok` for this.

**Note**: You'll need an `ngrok` account and an authtoken. You can get one from [ngrok's website](https://ngrok.com/). Add your authtoken to Colab's secrets manager (the '🔑' icon in the left panel) with the name `NGROK_AUTH_TOKEN`.

In [11]:
print('Installing ngrok...')
!pip install pyngrok --quiet

import os
from pyngrok import ngrok, conf
import time
from google.colab import userdata

# Get ngrok authtoken from Colab secrets
NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')

if NGROK_AUTH_TOKEN is None:
    raise ValueError("NGROK_AUTH_TOKEN not found in Colab secrets. Please add it.")

# Authenticate ngrok
conf.get_default().auth_token = NGROK_AUTH_TOKEN

# Ensure any previous ngrok processes are killed
!killall ngrok > /dev/null 2>&1 || true # Suppress output and ignore if no processes are found
ngrok.kill() # Kill pyngrok-managed tunnels
time.sleep(2) # Give ngrok a moment to shut down and release resources

# Disconnect any lingering tunnels reported by pyngrok
try:
    for tunnel in ngrok.get_tunnels():
        print(f"Disconnecting existing tunnel: {tunnel.public_url}")
        ngrok.disconnect(tunnel.public_url)
    print("Finished disconnecting existing ngrok tunnels.")
except Exception as e:
    print(f"Error during ngrok tunnel disconnection: {e}")

time.sleep(2) # Additional pause after disconnection

# Open a tunnel to the Ollama port (11434)
print('Starting ngrok tunnel...')
public_url = ngrok.connect(11434)

print(f"Ollama public URL: {public_url}")
print("You can now access Ollama from your local machine using this URL.")
print("Example: `ollama run llama2 --host {public_url}` on your local terminal")

Installing ngrok...
Finished disconnecting existing ngrok tunnels.
Starting ngrok tunnel...
Ollama public URL: NgrokTunnel: "https://boss-refurnish-agency.ngrok-free.dev" -> "http://localhost:11434"
You can now access Ollama from your local machine using this URL.
Example: `ollama run llama2 --host {public_url}` on your local terminal


In [12]:
%%bash
curl -X POST https://boss-refurnish-agency.ngrok-free.dev/api/generate -d '{
  "model": "qwen:0.5b",
  "prompt": "What is the fastest land animal?",
  "stream": false
}'

{"model":"qwen:0.5b","created_at":"2026-04-15T09:25:52.163325423Z","response":"The fastest land animal is the rhinoceros, which can run at speeds of up to 45 miles per hour (72 kilometers per hour)) during mating season.","done":true,"done_reason":"stop","context":[151644,872,198,3838,374,279,25648,4268,9864,30,151645,198,151644,77091,198,785,25648,4268,9864,374,279,21669,258,509,6264,11,892,646,1598,518,24722,315,705,311,220,19,20,8756,817,6460,220,7,22,17,40568,817,6460,593,2337,72119,3200,13],"total_duration":13753465541,"load_duration":667474361,"prompt_eval_count":15,"prompt_eval_duration":12786870310,"eval_count":37,"eval_duration":205605949}

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   749  100   656  100    93     46      6  0:00:15  0:00:14  0:00:01   164


### 6. (Optional) Cleanup

*(Note: Make sure to run this only after you are done testing the API via ngrok!)*

If you want to stop the Ollama server, you can kill the process using the process ID.

In [ ]:
# To stop the Ollama server, you can kill the process.
# Be careful with killing processes, ensure it's the correct one.
# The 'ollama_process' variable holds the Popen object.
if 'ollama_process' in locals() and ollama_process.poll() is None:
    ollama_process.terminate() # or .kill() for a more forceful termination
    print("Ollama server terminated.")
else:
    print("Ollama server not found or already stopped.")